# LICE-Guided Model Refinement and Ablation Evaluation

This notebook trains the baseline and seven ablation configurations on
the fixed training split, evaluates them on the untouched test split,
saves full-precision machine-readable outputs, and performs paired
McNemar comparisons. It is the main quick-run notebook for this release.

Run all cells from top to bottom in Google Colab. The preceding notebooks
document and regenerate the preparation, split, OOF, LIME/DiCE, and
sample-weight stages.


## 1. Connect Google Drive and install the environment


In [ ]:
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path

PROJECT_ROOT = Path(
    "/content/drive/MyDrive/Research/"
    "LICE_Guided_Model_Refinement_Release_V3"
)
assert PROJECT_ROOT.exists(), f"Release folder not found: {PROJECT_ROOT}"

from importlib.metadata import PackageNotFoundError, version
import os
import sys

PINNED_BINARY_STACK = {
    "numpy": ("numpy", "2.5.1"),
    "pandas": ("pandas", "2.2.3"),
    "scikit-learn": ("sklearn", "1.5.2"),
    "scipy": ("scipy", "1.18.0"),
    "statsmodels": ("statsmodels", "0.14.4"),
}

def installed_version(distribution_name):
    try:
        return version(distribution_name)
    except PackageNotFoundError:
        return None

restart_required = any(
    installed_version(distribution) != required
    or (
        module in sys.modules
        and getattr(sys.modules[module], "__version__", None) != required
    )
    for distribution, (module, required) in PINNED_BINARY_STACK.items()
)

%pip install -q -r "{PROJECT_ROOT / 'requirements.txt'}"

if restart_required:
    print(
        "Pinned packages were installed. Colab will restart now. "
        "After it reconnects, run this setup cell once more and "
        "then continue to the next cell.",
        flush=True,
    )
    os.kill(os.getpid(), 9)

print(f"Release folder: {PROJECT_ROOT}")


## 2. Environment and reproducibility configuration


In [1]:
from pathlib import Path
from importlib.metadata import version
import hashlib
import json
import os
import platform
import re
import sys

import joblib
import numpy as np
import pandas as pd
import scipy
import sklearn
import statsmodels

from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import (
    accuracy_score,
    cohen_kappa_score,
    confusion_matrix,
    f1_score,
    matthews_corrcoef,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split
from statsmodels.stats.contingency_tables import mcnemar

try:
    from IPython.display import display
except ImportError:
    display = print

DRIVE_RELEASE_ROOT = Path(
    "/content/drive/MyDrive/Research/"
    "LICE_Guided_Model_Refinement_Release_V3"
)
project_candidates = [
    Path.cwd(),
    Path.cwd().parent,
    DRIVE_RELEASE_ROOT,
]
PROJECT_ROOT = next(
    (
        candidate.resolve()
        for candidate in project_candidates
        if (candidate / "data" / "diabetes_brfss2015_prepared.csv").exists()
    ),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Project root not found. Run from the repository root or update "
        "DRIVE_RELEASE_ROOT in this cell."
    )

DATA_PATH = PROJECT_ROOT / "data" / "diabetes_brfss2015_prepared.csv"
WEIGHTS_PATH = PROJECT_ROOT / "input_artifacts" / "lice_sample_weights.csv.gz"
RESULTS_DIR = PROJECT_ROOT / "results"
MODELS_DIR = PROJECT_ROOT / "models"
RESULTS_DIR.mkdir(exist_ok=True)
MODELS_DIR.mkdir(exist_ok=True)

EXPECTED_VERSIONS = {
    "joblib": "1.5.3",
    "numpy": "2.5.1",
    "pandas": "2.2.3",
    "scikit-learn": "1.5.2",
    "scipy": "1.18.0",
    "statsmodels": "0.14.4",
}
installed_versions = {name: version(name) for name in EXPECTED_VERSIONS}
version_mismatches = {
    name: (EXPECTED_VERSIONS[name], installed)
    for name, installed in installed_versions.items()
    if installed != EXPECTED_VERSIONS[name]
}
if version_mismatches:
    raise RuntimeError(
        "Dependency mismatch. Install requirements.txt before execution: "
        f"{version_mismatches}"
    )

RANDOM_SEED = 42
TEST_SIZE = 0.20
DECISION_THRESHOLD = 0.50
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 220)
pd.set_option("display.precision", 15)

print("Environment verified.")
display(pd.DataFrame(
    [{"Package": name, "Version": value} for name, value in installed_versions.items()]
))


Environment verified.
        Package Version
0        joblib   1.5.3
1         numpy   2.5.1
2        pandas   2.2.3
3  scikit-learn   1.5.2
4         scipy  1.18.0
5   statsmodels  0.14.4


## 3. Load and validate the prepared dataset


In [2]:
EXPECTED_DATA_SHA256 = "cea4e25cd6304a5f35f5f71cd1b374bef9613ecb9e1ff2a9b2056c7a3d8b7cc8"
EXPECTED_COLUMNS = [
    "Outcome", "HighBP", "HighChol", "CholCheck", "BMI", "Smoker", "Stroke",
    "HeartDiseaseorAttack", "PhysActivity", "Fruits", "Veggies",
    "HvyAlcoholConsump", "AnyHealthcare", "NoDocbcCost", "GenHlth",
    "MentHlth", "PhysHlth", "DiffWalk", "Sex", "Age", "Education", "Income",
]

def file_sha256(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()

assert file_sha256(DATA_PATH) == EXPECTED_DATA_SHA256
dataset = pd.read_csv(DATA_PATH)
assert dataset.shape == (69057, 22)
assert dataset.columns.tolist() == EXPECTED_COLUMNS
assert int(dataset.isna().sum().sum()) == 0

X = dataset.drop(columns=["Outcome"])
y = dataset["Outcome"].astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=TEST_SIZE,
    random_state=RANDOM_SEED,
)
y_train_array = y_train.to_numpy(dtype=int)
y_test_array = y_test.to_numpy(dtype=int)

assert X_train.shape == (55245, 21)
assert X_test.shape == (13812, 21)
assert y_train.value_counts().sort_index().to_dict() == {0: 27237, 1: 28008}
assert y_test.value_counts().sort_index().to_dict() == {0: 6723, 1: 7089}

dataset_summary = pd.DataFrame([
    {"Item": "Prepared dataset rows", "Value": len(dataset)},
    {"Item": "Predictor count", "Value": X.shape[1]},
    {"Item": "Training rows", "Value": len(X_train)},
    {"Item": "Test rows", "Value": len(X_test)},
    {"Item": "Random seed", "Value": RANDOM_SEED},
    {"Item": "Test fraction", "Value": TEST_SIZE},
    {"Item": "Decision threshold", "Value": DECISION_THRESHOLD},
])
display(dataset_summary)
display(pd.DataFrame({
    "Training": y_train.value_counts().sort_index(),
    "Test": y_test.value_counts().sort_index(),
}).rename_axis("Class"))


                    Item    Value
0  Prepared dataset rows  69057.0
1        Predictor count     21.0
2          Training rows  55245.0
3              Test rows  13812.0
4            Random seed     42.0
5          Test fraction      0.2
6     Decision threshold      0.5
       Training  Test
Class                
0         27237  6723
1         28008  7089


## 4. Load and validate the LICE-derived sample weights


In [3]:
EXPECTED_WEIGHTS_SHA256 = "812aa92982d1c1264efd82b821eb9e7baf9d35e900517cc674ad83e504d5d607"
assert file_sha256(WEIGHTS_PATH) == EXPECTED_WEIGHTS_SHA256

weights_table = pd.read_csv(WEIGHTS_PATH)
assert len(weights_table) == len(X_train)
assert np.array_equal(
    weights_table["Train_Position"].to_numpy(dtype=int),
    np.arange(len(X_train), dtype=int),
)
assert np.array_equal(
    weights_table["Source_Row_Index"].to_numpy(dtype=int),
    X_train.index.to_numpy(dtype=int),
)

sample_weights = {
    "Mild": weights_table["Mild"].to_numpy(dtype=float),
    "Balanced": weights_table["Balanced"].to_numpy(dtype=float),
    "High": weights_table["High"].to_numpy(dtype=float),
}

weight_summary = pd.DataFrame([
    {
        "Weight_Level": name,
        "Training_Rows": len(values),
        "Weighted_Rows": int(np.sum(values > 1.0)),
        "Mean_Weight": float(np.mean(values)),
        "Maximum_Weight": float(np.max(values)),
        "Unique_Weights": ", ".join(map(str, np.unique(values))),
    }
    for name, values in sample_weights.items()
])

assert weight_summary["Weighted_Rows"].tolist() == [7174, 7174, 7174]
display(weight_summary)


  Weight_Level  Training_Rows  Weighted_Rows        Mean_Weight  Maximum_Weight   Unique_Weights
0         Mild          55245           7174  1.014873744230247            1.15   1.0, 1.1, 1.15
1     Balanced          55245           7174  1.023254593175853            1.25  1.0, 1.15, 1.25
2         High          55245           7174  1.031635442121459            1.35   1.0, 1.2, 1.35


## 5. Fixed classifier configuration and evaluation utilities


In [4]:
MODEL_PARAMETERS = {
    "ccp_alpha": 0.0,
    "criterion": "friedman_mse",
    "init": None,
    "learning_rate": 0.1,
    "loss": "log_loss",
    "max_depth": 5,
    "max_features": None,
    "max_leaf_nodes": None,
    "min_impurity_decrease": 0.0,
    "min_samples_leaf": 1,
    "min_samples_split": 2,
    "min_weight_fraction_leaf": 0.0,
    "n_estimators": 100,
    "n_iter_no_change": None,
    "random_state": RANDOM_SEED,
    "subsample": 1.0,
    "tol": 0.0001,
    "validation_fraction": 0.1,
    "verbose": 0,
    "warm_start": False,
}

METRIC_COLUMNS = ["Accuracy", "AUC", "Recall", "Precision", "F1", "Kappa", "MCC"]

def calculate_metrics(y_true, y_prediction, y_probability):
    return {
        "Accuracy": accuracy_score(y_true, y_prediction),
        "AUC": roc_auc_score(y_true, y_probability),
        "Recall": recall_score(y_true, y_prediction, pos_label=1, zero_division=0),
        "Precision": precision_score(
            y_true, y_prediction, pos_label=1, zero_division=0
        ),
        "F1": f1_score(y_true, y_prediction, pos_label=1, zero_division=0),
        "Kappa": cohen_kappa_score(y_true, y_prediction),
        "MCC": matthews_corrcoef(y_true, y_prediction),
    }

def train_and_evaluate(model_name, X_train_fit, X_test_eval, weights=None):
    model = GradientBoostingClassifier(**MODEL_PARAMETERS)
    if weights is None:
        model.fit(X_train_fit, y_train_array)
        weight_level = "None"
        weighted_rows = 0
        mean_weight = 1.0
        max_weight = 1.0
    else:
        assert len(weights) == len(X_train_fit)
        model.fit(X_train_fit, y_train_array, sample_weight=weights)
        weight_level = model_name.split("-")[-1]
        weighted_rows = int(np.sum(weights > 1.0))
        mean_weight = float(np.mean(weights))
        max_weight = float(np.max(weights))

    probabilities = model.predict_proba(X_test_eval)[:, 1]
    predictions = (probabilities >= DECISION_THRESHOLD).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_test_array, predictions).ravel()

    result = {
        "Model": model_name,
        "Training_Samples": len(X_train_fit),
        "Test_Samples": len(X_test_eval),
        "Weight_Level": weight_level,
        "Weighted_Training_Samples": weighted_rows,
        "Mean_Training_Weight": mean_weight,
        "Maximum_Training_Weight": max_weight,
        "TN": int(tn),
        "FP": int(fp),
        "FN": int(fn),
        "TP": int(tp),
        **calculate_metrics(y_test_array, predictions, probabilities),
    }
    return model, result, predictions, probabilities

display(pd.DataFrame(
    [{"Parameter": key, "Value": value} for key, value in MODEL_PARAMETERS.items()]
))


                   Parameter         Value
0                  ccp_alpha           0.0
1                  criterion  friedman_mse
2                       init          None
3              learning_rate           0.1
4                       loss      log_loss
5                  max_depth             5
6               max_features          None
7             max_leaf_nodes          None
8      min_impurity_decrease           0.0
9           min_samples_leaf             1
10         min_samples_split             2
11  min_weight_fraction_leaf           0.0
12              n_estimators           100
13          n_iter_no_change          None
14              random_state            42
15                 subsample           1.0
16                       tol        0.0001
17       validation_fraction           0.1
18                   verbose             0
19                warm_start         False


## 6. Construct the LICE-derived interaction features


In [5]:
INTERACTION_DEFINITIONS = {
    "HighBP_HighChol": ("HighBP", "HighChol"),
    "HighBP_GenHlth": ("HighBP", "GenHlth"),
    "HighChol_GenHlth": ("HighChol", "GenHlth"),
    "BMI_GenHlth": ("BMI", "GenHlth"),
}

def add_lice_interactions(features):
    enriched = features.copy()
    for output_feature, (left_feature, right_feature) in INTERACTION_DEFINITIONS.items():
        enriched[output_feature] = (
            enriched[left_feature] * enriched[right_feature]
        )
    return enriched

X_train_interactions = add_lice_interactions(X_train)
X_test_interactions = add_lice_interactions(X_test)
assert X_train_interactions.shape == (55245, 25)
assert X_test_interactions.shape == (13812, 25)

interaction_summary = pd.DataFrame([
    {
        "Interaction_Feature": output_feature,
        "Definition": f"{left_feature} * {right_feature}",
    }
    for output_feature, (left_feature, right_feature)
    in INTERACTION_DEFINITIONS.items()
])
display(interaction_summary)


  Interaction_Feature          Definition
0     HighBP_HighChol   HighBP * HighChol
1      HighBP_GenHlth    HighBP * GenHlth
2    HighChol_GenHlth  HighChol * GenHlth
3         BMI_GenHlth       BMI * GenHlth


## 7. Train and evaluate the ablation models

| Public model name | Experimental configuration |
|---|---|
| Baseline | Fixed GBDT; original features; no LICE-derived weights |
| M1-Interactions | Fixed GBDT; LICE-derived interactions only |
| M2-Mild / Balanced / High | Fixed GBDT; original features; LICE-derived sample weights |
| M3-Mild / Balanced / High | Fixed GBDT; LICE-derived interactions and sample weights |


In [6]:
MODEL_CONFIGURATIONS = [
    ("Baseline", X_train, X_test, None),
    ("M1-Interactions", X_train_interactions, X_test_interactions, None),
    ("M2-Mild", X_train, X_test, sample_weights["Mild"]),
    ("M2-Balanced", X_train, X_test, sample_weights["Balanced"]),
    ("M2-High", X_train, X_test, sample_weights["High"]),
    ("M3-Mild", X_train_interactions, X_test_interactions, sample_weights["Mild"]),
    (
        "M3-Balanced",
        X_train_interactions,
        X_test_interactions,
        sample_weights["Balanced"],
    ),
    (
        "M3-High",
        X_train_interactions,
        X_test_interactions,
        sample_weights["High"],
    ),
]

trained_models = {}
model_predictions = {}
model_probabilities = {}
result_rows = []

for model_name, train_features, test_features, weights in MODEL_CONFIGURATIONS:
    print(f"Training {model_name} ...")
    model, result, predictions, probabilities = train_and_evaluate(
        model_name,
        train_features,
        test_features,
        weights,
    )
    trained_models[model_name] = model
    model_predictions[model_name] = predictions
    model_probabilities[model_name] = probabilities
    result_rows.append(result)

metrics_full_precision = pd.DataFrame(result_rows)
metrics_full_precision.to_csv(
    RESULTS_DIR / "metrics_full_precision.csv",
    index=False,
    float_format="%.17g",
)
metrics_full_precision[
    ["Model", "TN", "FP", "FN", "TP"]
].to_csv(
    RESULTS_DIR / "confusion_matrices.csv",
    index=False,
)

prediction_output = pd.DataFrame({
    "Source_Row_Index": X_test.index.to_numpy(dtype=int),
    "y_true": y_test_array,
})
for model_name, _, _, _ in MODEL_CONFIGURATIONS:
    column_prefix = re.sub(r"[^a-z0-9]+", "_", model_name.lower()).strip("_")
    prediction_output[f"{column_prefix}_prediction"] = model_predictions[model_name]
    prediction_output[f"{column_prefix}_probability"] = model_probabilities[model_name]
prediction_output.to_csv(
    RESULTS_DIR / "predictions_all_models.csv.gz",
    index=False,
    float_format="%.17g",
    compression={"method": "gzip", "compresslevel": 9, "mtime": 0},
)

joblib.dump(trained_models["M3-Balanced"], MODELS_DIR / "M3_Balanced.joblib")
joblib.dump(trained_models["M3-High"], MODELS_DIR / "M3_High.joblib")

display(metrics_full_precision)


Training Baseline ...
Training M1-Interactions ...
Training M2-Mild ...
Training M2-Balanced ...
Training M2-High ...
Training M3-Mild ...
Training M3-Balanced ...
Training M3-High ...
             Model  Training_Samples  Test_Samples Weight_Level  Weighted_Training_Samples  Mean_Training_Weight  Maximum_Training_Weight    TN    FP    FN    TP           Accuracy                AUC  \
0         Baseline             55245         13812         None                          0     1.000000000000000                     1.00  4767  1956  1417  5672  0.755792064871127  0.830959234502311   
1  M1-Interactions             55245         13812         None                          0     1.000000000000000                     1.00  4765  1958  1393  5696  0.757384882710686  0.831701072194716   
2          M2-Mild             55245         13812         Mild                       7174     1.014873744230247                     1.15  4666  2057  1326  5763  0.755068056762236  0.830215340550092   
3  

## 8. Full-precision comparison and candidate ranking


In [7]:
baseline = metrics_full_precision.loc[
    metrics_full_precision["Model"] == "Baseline"
].iloc[0]

comparison_rows = []
for _, row in metrics_full_precision.iterrows():
    comparison = row.to_dict()
    for metric in METRIC_COLUMNS:
        comparison[f"Delta_{metric}"] = float(row[metric] - baseline[metric])
    comparison["FN_Reduction"] = int(baseline["FN"] - row["FN"])
    comparison["FP_Increase"] = int(row["FP"] - baseline["FP"])
    comparison["Relative_FN_Reduction_Percentage"] = (
        (baseline["FN"] - row["FN"]) / baseline["FN"] * 100.0
    )
    comparison["Relative_FP_Increase_Percentage"] = (
        (row["FP"] - baseline["FP"]) / baseline["FP"] * 100.0
    )
    comparison_rows.append(comparison)

comparison_full_precision = pd.DataFrame(comparison_rows)
comparison_full_precision.to_csv(
    RESULTS_DIR / "comparison_vs_baseline_full_precision.csv",
    index=False,
    float_format="%.17g",
)

rankings = comparison_full_precision[
    comparison_full_precision["Model"] != "Baseline"
].copy()
rankings["Rank_by_F1"] = (
    rankings["F1"].rank(method="min", ascending=False).astype(int)
)
rankings["Rank_by_Recall"] = (
    rankings["Recall"].rank(method="min", ascending=False).astype(int)
)
rankings["Rank_by_FN_Reduction"] = (
    rankings["FN_Reduction"].rank(method="min", ascending=False).astype(int)
)
rankings = rankings.sort_values(
    ["Rank_by_F1", "Rank_by_FN_Reduction", "Rank_by_Recall"]
)
rankings.to_csv(
    RESULTS_DIR / "rankings_full_precision.csv",
    index=False,
    float_format="%.17g",
)

candidate_summary = comparison_full_precision[
    comparison_full_precision["Model"].isin(["Baseline", "M3-Balanced", "M3-High"])
].copy()
candidate_summary.to_csv(
    RESULTS_DIR / "candidate_summary_full_precision.csv",
    index=False,
    float_format="%.17g",
)

display(comparison_full_precision)
display(rankings)
display(candidate_summary)


             Model  Training_Samples  Test_Samples Weight_Level  Weighted_Training_Samples  Mean_Training_Weight  Maximum_Training_Weight    TN    FP    FN    TP           Accuracy                AUC  \
0         Baseline             55245         13812         None                          0     1.000000000000000                     1.00  4767  1956  1417  5672  0.755792064871127  0.830959234502311   
1  M1-Interactions             55245         13812         None                          0     1.000000000000000                     1.00  4765  1958  1393  5696  0.757384882710686  0.831701072194716   
2          M2-Mild             55245         13812         Mild                       7174     1.014873744230247                     1.15  4666  2057  1326  5763  0.755068056762236  0.830215340550092   
3      M2-Balanced             55245         13812     Balanced                       7174     1.023254593175853                     1.25  4621  2102  1265  5824  0.756226469736461  0.8302

## 9. Paired McNemar comparisons


In [8]:
def run_mcnemar(y_true, baseline_prediction, candidate_prediction, comparison, scope):
    baseline_correct = baseline_prediction == y_true
    candidate_correct = candidate_prediction == y_true

    both_correct = int(np.sum(baseline_correct & candidate_correct))
    baseline_correct_candidate_wrong = int(
        np.sum(baseline_correct & ~candidate_correct)
    )
    baseline_wrong_candidate_correct = int(
        np.sum(~baseline_correct & candidate_correct)
    )
    both_wrong = int(np.sum(~baseline_correct & ~candidate_correct))

    result = mcnemar(
        [
            [both_correct, baseline_correct_candidate_wrong],
            [baseline_wrong_candidate_correct, both_wrong],
        ],
        exact=False,
        correction=True,
    )
    if baseline_wrong_candidate_correct > baseline_correct_candidate_wrong:
        direction = "Candidate corrected more baseline errors than it introduced"
    elif baseline_wrong_candidate_correct < baseline_correct_candidate_wrong:
        direction = "Candidate introduced more errors than it corrected"
    else:
        direction = "Equal disagreement counts"

    return {
        "Scope": scope,
        "Comparison": comparison,
        "Both_Correct": both_correct,
        "Baseline_Correct_Candidate_Wrong": baseline_correct_candidate_wrong,
        "Baseline_Wrong_Candidate_Correct": baseline_wrong_candidate_correct,
        "Both_Wrong": both_wrong,
        "McNemar_Statistic": float(result.statistic),
        "p_value": float(result.pvalue),
        "Significant_at_0.05": bool(result.pvalue < 0.05),
        "Direction": direction,
    }

mcnemar_rows = []
positive_mask = y_test_array == 1
for candidate_name in ["M3-Balanced", "M3-High"]:
    mcnemar_rows.append(
        run_mcnemar(
            y_test_array,
            model_predictions["Baseline"],
            model_predictions[candidate_name],
            f"Baseline vs {candidate_name}",
            "Full test set",
        )
    )
    mcnemar_rows.append(
        run_mcnemar(
            y_test_array[positive_mask],
            model_predictions["Baseline"][positive_mask],
            model_predictions[candidate_name][positive_mask],
            f"Baseline vs {candidate_name}",
            "Actual positive cases only",
        )
    )

mcnemar_full_precision = pd.DataFrame(mcnemar_rows)
mcnemar_full_precision.to_csv(
    RESULTS_DIR / "mcnemar_results_full_precision.csv",
    index=False,
    float_format="%.17g",
)
display(mcnemar_full_precision)


                        Scope               Comparison  Both_Correct  Baseline_Correct_Candidate_Wrong  Baseline_Wrong_Candidate_Correct  Both_Wrong    McNemar_Statistic                p_value  Significant_at_0.05  \
0               Full test set  Baseline vs M3-Balanced         10194                               245                               263        3110    0.568897637795276  4.506972985378143e-01                False   
1  Actual positive cases only  Baseline vs M3-Balanced          5632                                40                               212        1205  116.035714285714292  4.668137640136469e-27                 True   
2               Full test set      Baseline vs M3-High         10125                               314                               304        3069    0.131067961165049  7.173273000417576e-01                False   
3  Actual positive cases only      Baseline vs M3-High          5620                                52                              

## 10. Internal consistency checks and run record

The checks below recalculate confusion counts and all reported
metrics directly from the case-level predictions produced in this
run. They do not compare against a hard-coded table.


In [ ]:
verification_rows = []
for model_name, prediction in model_predictions.items():
    probability = model_probabilities[model_name]
    tn, fp, fn, tp = confusion_matrix(
        y_test_array, prediction
    ).ravel()
    recalculated = {
        "TN": int(tn), "FP": int(fp), "FN": int(fn), "TP": int(tp),
        "Accuracy": accuracy_score(y_test_array, prediction),
        "Recall": recall_score(y_test_array, prediction),
        "Precision": precision_score(y_test_array, prediction),
        "F1": f1_score(y_test_array, prediction),
        "MCC": matthews_corrcoef(y_test_array, prediction),
        "Kappa": cohen_kappa_score(y_test_array, prediction),
        "AUC": roc_auc_score(y_test_array, probability),
    }
    saved = metrics_full_precision.loc[
        metrics_full_precision.Model == model_name
    ].iloc[0]
    for name in ["TN", "FP", "FN", "TP"]:
        assert int(saved[name]) == recalculated[name]
    for name in METRIC_COLUMNS:
        assert np.isclose(
            float(saved[name]), recalculated[name],
            rtol=0.0, atol=1e-15,
        ), f"{model_name} {name} differs after recalculation"
    verification_rows.append({
        "Model": model_name,
        "Counts_recalculated": True,
        "Metrics_recalculated": True,
    })

verification = pd.DataFrame(verification_rows)
verification.to_csv(
    RESULTS_DIR / "internal_consistency_checks.csv", index=False
)

manifest = {
    "experiment": "LICE-guided model refinement and ablation evaluation",
    "scope": (
        "Fixed prepared dataset and LICE-derived weight artifacts "
        "through untouched test-set evaluation"
    ),
    "python_version": platform.python_version(),
    "package_versions": installed_versions,
    "random_seed": RANDOM_SEED,
    "test_size": TEST_SIZE,
    "decision_threshold": DECISION_THRESHOLD,
    "model_parameters": MODEL_PARAMETERS,
    "dataset": {
        "file": str(DATA_PATH.relative_to(PROJECT_ROOT)),
        "sha256": file_sha256(DATA_PATH),
        "shape": list(dataset.shape),
    },
    "sample_weights": {
        "file": str(WEIGHTS_PATH.relative_to(PROJECT_ROOT)),
        "sha256": file_sha256(WEIGHTS_PATH),
        "training_rows": len(weights_table),
    },
    "selected_pretrained_models": {
        "M3-Balanced": {
            "file": "models/M3_Balanced.joblib",
            "sha256": file_sha256(
                MODELS_DIR / "M3_Balanced.joblib"
            ),
        },
        "M3-High": {
            "file": "models/M3_High.joblib",
            "sha256": file_sha256(MODELS_DIR / "M3_High.joblib"),
        },
    },
    "precision_policy": (
        "No intermediate rounding; CSV floating-point values use "
        "17 significant digits; manuscript values are rounded to "
        "four decimal places only for presentation."
    ),
    "consistency_check": (
        "Confusion counts and metrics recalculated from case-level "
        "predictions generated in this run."
    ),
}
with (RESULTS_DIR / "run_manifest.json").open(
    "w", encoding="utf-8"
) as stream:
    json.dump(manifest, stream, indent=2)

display(verification)
print("Evaluation completed and all result files saved successfully.")
